In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
batch_size = 1000
# batch_size = None

opt = torch_numopt.NewtonRaphson(
    params=model.parameters(),
    model=model,
    lr=1,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    batch_size=batch_size
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.635820209980011
epoch:  1, loss: 0.44769489765167236
epoch:  2, loss: 0.34503576159477234
epoch:  3, loss: 0.09576347470283508
epoch:  4, loss: 0.07986336201429367
epoch:  5, loss: 0.011672713793814182
epoch:  6, loss: 0.009000851772725582
epoch:  7, loss: 0.007485365495085716
epoch:  8, loss: 0.006063396111130714
epoch:  9, loss: 0.005279103294014931
epoch:  10, loss: 0.0047481851652264595
epoch:  11, loss: 0.004346050322055817
epoch:  12, loss: 0.004081808030605316
epoch:  13, loss: 0.003840771270915866
epoch:  14, loss: 0.00364674162119627
epoch:  15, loss: 0.003494806122034788
epoch:  16, loss: 0.003362621646374464
epoch:  17, loss: 0.0032657759729772806
epoch:  18, loss: 0.0031792265363037586
epoch:  19, loss: 0.003093797480687499
epoch:  20, loss: 0.003015116322785616
epoch:  21, loss: 0.0029536383226513863
epoch:  22, loss: 0.0029010730795562267
epoch:  23, loss: 0.0028514908626675606
epoch:  24, loss: 0.0028112786822021008
epoch:  25, loss: 0.0027675100136548

In [12]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9458752870559692
Test metrics:  R2 = 0.7251042723655701


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
batch_size = 1000
# batch_size = None

opt = torch_numopt.NewtonRaphson(
    params=model.parameters(),
    model=model,
    lr=1,
    c1=1e-4,
    tau=0.1,
    daming=True,
    mu=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    batch_size=batch_size
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3818720579147339
epoch:  1, loss: 0.24467326700687408
epoch:  2, loss: 0.20633135735988617
epoch:  3, loss: 0.16920290887355804
epoch:  4, loss: 0.1348150223493576
epoch:  5, loss: 0.11425332725048065
epoch:  6, loss: 0.034096673130989075
epoch:  7, loss: 0.02972656674683094
epoch:  8, loss: 0.024586424231529236
epoch:  9, loss: 0.008996373973786831
epoch:  10, loss: 0.005396274849772453
epoch:  11, loss: 0.003585302969440818
epoch:  12, loss: 0.0030071819201111794
epoch:  13, loss: 0.002544273156672716
epoch:  14, loss: 0.002262270776554942
epoch:  15, loss: 0.0020341265480965376
epoch:  16, loss: 0.0018696835031732917
epoch:  17, loss: 0.0017474155174568295
epoch:  18, loss: 0.0016447624657303095
epoch:  19, loss: 0.0015841188142076135
epoch:  20, loss: 0.0015042441664263606
epoch:  21, loss: 0.0014458487275987864
epoch:  22, loss: 0.0014018341898918152
epoch:  23, loss: 0.0013692021602764726
epoch:  24, loss: 0.0013304209569469094
epoch:  25, loss: 0.0012960245367

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9738389849662781
Test metrics:  R2 = 0.920015811920166
